# Read-level bootstrapping with fastQpick on a real RNA-seq library

This notebook demonstrates FASTQ sampling **with replacement**, the capability that
distinguishes `fastQpick` from common subsampling tools such as `seqtk sample`. Sampling
reads with replacement turns a single FASTQ file into an arbitrary number of bootstrap
replicates, which lets one attach uncertainty estimates to any downstream quantity
computed from the reads without re-running the wet-lab experiment.

A real paired-end *Saccharomyces cerevisiae* RNA-seq library (SRA accession
`SRR453566`, Nookaew et al. 2012) is quantified with `kallisto`. `fastQpick` then draws
$B$ full-size bootstrap replicates of the read pair (`fraction=1.0`, `replacement=True`,
mates grouped with `file_group_size=2`), each replicate is re-quantified, and the
bootstrap distribution of the estimated transcript proportions is compared against the
analytic multinomial sampling error $\sqrt{\hat{p}(1-\hat{p})/N}$. The bootstrap standard
errors recover the analytic errors, confirming that read-level resampling with
replacement captures the sampling uncertainty of the quantification on real data.

> A self-contained simulated version of this analysis (with barcoded reads of known
> abundance) is kept in `intro.ipynb`.

**Compute note.** Generating and re-quantifying $B=200$ full-size replicates is the
expensive step. The manuscript runs it 8-way parallel with `realdata/run_one.py` and
`realdata/drive.sh`. The serial loop below reproduces the same result; set `B` smaller
for a quick run.

In [ ]:
import os
import glob
import json
import shutil
import subprocess

import numpy as np
import matplotlib.pyplot as plt
import pyfastx

from fastQpick import fastQpick

WORK = os.path.abspath("realdata")
FIG_DIR = os.path.abspath("figures")
os.makedirs(WORK, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

KALLISTO = os.path.join(WORK, "kallisto", "kallisto")
IDX = os.path.join(WORK, "yeast.idx")
M1 = os.path.join(WORK, "data", "SRR453566_1.fastq.gz")
M2 = os.path.join(WORK, "data", "SRR453566_2.fastq.gz")
ACCESSION = "SRR453566"

## 1. Reference, reads, and index

A precompiled `kallisto` binary, the Ensembl R64-1-1 yeast cDNA reference, and the paired
FASTQ files are downloaded if not already present, and a `kallisto` index is built. The
two mate files contain equal numbers of reads, which `fastQpick` requires for grouping.

In [ ]:
def sh(cmd):
    subprocess.run(cmd, shell=True, check=True)

# kallisto binary
if not os.path.exists(KALLISTO):
    sh(f"cd {WORK} && wget -q https://github.com/pachterlab/kallisto/releases/download/"
       f"v0.51.1/kallisto_linux-v0.51.1.tar.gz -O kallisto.tar.gz && tar xzf kallisto.tar.gz")

# yeast cDNA reference + index
cdna = os.path.join(WORK, "cdna.fa.gz")
if not os.path.exists(cdna):
    sh(f"wget -q http://ftp.ensembl.org/pub/release-110/fasta/saccharomyces_cerevisiae/"
       f"cdna/Saccharomyces_cerevisiae.R64-1-1.cdna.all.fa.gz -O {cdna}")
if not os.path.exists(IDX):
    sh(f"{KALLISTO} index -i {IDX} {cdna}")

# paired reads
os.makedirs(os.path.join(WORK, "data"), exist_ok=True)
for r, path in [(1, M1), (2, M2)]:
    if not os.path.exists(path):
        sh(f"wget -q https://ftp.sra.ebi.ac.uk/vol1/fastq/SRR453/{ACCESSION}/"
           f"{ACCESSION}_{r}.fastq.gz -O {path}")

N_READS = sum(1 for _ in pyfastx.Fastx(M1))
print("reads per mate:", N_READS)

## 2. Baseline quantification

The original library is quantified once with `kallisto`. The estimated counts give the
point-estimate transcript proportions $\hat{p}_t$ and the number of pseudoaligned reads
$N$, which set the analytic multinomial sampling error.

In [ ]:
base_dir = os.path.join(WORK, "kout_baseline")
sh(f"{KALLISTO} quant -i {IDX} -o {base_dir} -t 4 {M1} {M2}")

targets = np.loadtxt(os.path.join(base_dir, "abundance.tsv"), skiprows=1, usecols=0, dtype=str)
baseline = np.loadtxt(os.path.join(base_dir, "abundance.tsv"), skiprows=1, usecols=3)
info = json.load(open(os.path.join(base_dir, "run_info.json")))
print("transcripts:", len(targets), " pseudoaligned:", info["n_pseudoaligned"],
      f"({info['p_pseudoaligned']}%)")

## 3. Bootstrap replicates with replacement

Each replicate is a full-size resample of the original FASTQ pair drawn **with
replacement** (`fraction=1.0`, `replacement=True`). The two mate files are grouped
(`file_group_size=2`) so that paired reads are resampled together, keeping mates
synchronized. Counting the reads happens only once: the resulting `fastq_to_length_dict`
is reused across replicates. Each replicate is re-quantified and then deleted to bound
disk use.

In [ ]:
B = 200  # reduce for a quick run
length_dict = {M1: N_READS, M2: N_READS}
counts_dir = os.path.join(WORK, "counts_nb")
os.makedirs(counts_dir, exist_ok=True)

def quantify_replicate(seed):
    tmp = os.path.join(WORK, f"tmp_nb/seed{seed}")
    rep, kq = os.path.join(tmp, "rep"), os.path.join(tmp, "kq")
    fastQpick(input_files=[M1, M2], fraction=1.0, seeds=seed, output_dir=rep,
              file_group_size=2, replacement=True, overwrite=True, verbose=False,
              fastq_to_length_dict=length_dict)
    r1 = os.path.join(rep, "SRR453566_1.fastq")
    r2 = os.path.join(rep, "SRR453566_2.fastq")
    subprocess.run([KALLISTO, "quant", "-i", IDX, "-o", kq, "-t", "4", r1, r2],
                   check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    est = np.loadtxt(os.path.join(kq, "abundance.tsv"), skiprows=1, usecols=3)
    shutil.rmtree(tmp, ignore_errors=True)
    return est

R = np.vstack([quantify_replicate(b) for b in range(1, B + 1)])
print("bootstrap count matrix (replicates x transcripts):", R.shape)

## 4. Bootstrap standard error vs. analytic multinomial error

For a proportion estimated from $N$ reads, the analytic multinomial sampling error is
$\sqrt{\hat{p}(1-\hat{p})/N}$. The read-level bootstrap should recover this directly from
the data. The comparison is restricted to expressed transcripts (at least 100 estimated
counts) so that it is not dominated by near-zero-count Monte Carlo noise.

In [ ]:
P = R / R.sum(axis=1, keepdims=True)
N = baseline.sum()
p_hat = baseline / N
boot_se = P.std(axis=0, ddof=1)
analytic_se = np.sqrt(p_hat * (1 - p_hat) / N)

expressed = baseline >= 100
rel = np.abs(boot_se[expressed] - analytic_se[expressed]) / analytic_se[expressed]
ratio = (boot_se[expressed] / analytic_se[expressed]).mean()
print(f"expressed transcripts            : {expressed.sum()}")
print(f"median |boot-analytic|/analytic  : {np.median(rel):.1%}")
print(f"mean boot/analytic ratio         : {ratio:.4f}")

In [ ]:
THETA = 5.0  # detection threshold (estimated counts)
detect_prob = (R >= THETA).mean(axis=0)              # bootstrap detection prob. pi_t
persist = detect_prob >= 0.95
fragile = (detect_prob > 0.05) & (detect_prob < 0.95)
print(f"fragile calls (0.05<pi<0.95)     : {int(fragile.sum())}")
print(f"  called at baseline, may vanish : {int((fragile & (baseline >= THETA)).sum())}")
print(f"  absent at baseline, may appear : {int((fragile & (baseline < THETA)).sum())}")

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(16.5, 4.3))

# Panel A: standardized bootstrap deviations pooled across expressed transcripts.
z = ((P[:, expressed] - p_hat[expressed]) / analytic_se[expressed]).ravel()
ax1.hist(z, bins=80, density=True, color="#4C72B0", alpha=0.6,
         label=f"pooled $z$  (std={z.std(ddof=1):.2f})")
xs = np.linspace(-4, 4, 400)
ax1.plot(xs, np.exp(-0.5 * xs**2) / np.sqrt(2*np.pi), color="#C44E52", lw=2,
         label=r"$\mathcal{N}(0,1)$")
ax1.set_xlim(-4, 4)
ax1.set_xlabel(r"standardized deviation  $(\hat{p}^{(b)}-\hat{p})/$SE$_{\mathrm{analytic}}$")
ax1.set_ylabel("density")
ax1.set_title("A. Standardized bootstrap deviations vs. $\\mathcal{N}(0,1)$")
ax1.legend(frameon=False, fontsize=9)

# Panel B: bootstrap SE vs analytic SE across all expressed transcripts.
ase, bse = analytic_se[expressed], boot_se[expressed]
lo, hi = min(ase.min(), bse.min())*0.8, max(ase.max(), bse.max())*1.2
ax2.plot([lo, hi], [lo, hi], color="#888888", ls="--", lw=1, label="y = x")
sc = ax2.scatter(ase, bse, c=np.log10(baseline[expressed]), cmap="viridis", s=8, alpha=0.6)
ax2.set_xscale("log"); ax2.set_yscale("log")
ax2.set_xlim(lo, hi); ax2.set_ylim(lo, hi)
ax2.set_xlabel(r"analytic SE  $\sqrt{\hat{p}(1-\hat{p})/N}$")
ax2.set_ylabel("bootstrap SE")
ax2.set_title("B. Bootstrap SE recovers analytic SE")
fig.colorbar(sc, ax=ax2).set_label(r"$\log_{10}$ est. counts", fontsize=8)
ax2.legend(frameon=False, fontsize=9, loc="upper left")

# Panel C: bootstrap detection probability vs. point estimate for low-abundance
# transcripts. Detection is a discrete presence/absence event, so the Gaussian SE
# of panel B does not describe it; the bootstrap instead gives a per-transcript
# detection probability that flags which low-abundance calls persist and which
# are driven by a handful of reads and go away under resampling.
# Persist and fragile transcripts are plotted in full so the legend counts match
# the totals reported in the text; rarely-called transcripts are restricted to the
# low-count regime, and zero counts are floored to the left edge of the log axis.
XFLOOR = 0.3
cat = np.where(persist, 0, np.where(fragile, 1, 2))
lowab = baseline > XFLOOR
colors = ["#55A868", "#DD8452", "#C44E52"]           # persist / fragile / rarely called
labels = ["persists ($\\pi\\geq0.95$)", "fragile ($0.05<\\pi<0.95$)",
          "rarely called ($\\pi\\leq0.05$)"]
for k in (0, 1, 2):                                  # persists, fragile, rarely called
    m = (cat == k) if k != 2 else ((cat == k) & lowab)
    xk = np.clip(baseline[m], XFLOOR, None)
    ax3.scatter(xk, detect_prob[m], s=10, alpha=0.6, color=colors[k],
                label=f"{labels[k]}  (n={int(m.sum())})")
ax3.set_xlim(left=XFLOOR * 0.9)
ax3.axvline(THETA, color="#888888", ls="--", lw=1)
ax3.text(THETA*1.1, 0.06, f"threshold $\\theta={THETA:g}$", rotation=90,
         va="bottom", ha="left", fontsize=8, color="#888888")
ax3.set_xscale("log")
ax3.set_ylim(-0.03, 1.03)
ax3.set_xlabel("point-estimate count $\\hat{c}_t$ (baseline)")
ax3.set_ylabel(r"bootstrap detection probability $\pi_t$")
ax3.set_title("C. Detection robustness of low-abundance calls")
ax3.legend(frameon=False, fontsize=8, loc="center right")

fig.tight_layout()
out = os.path.join(FIG_DIR, "bootstrap_realdata.png")
fig.savefig(out, dpi=200)
print("Saved", out)
plt.show()

## 5. Summary

Using `fastQpick` to resample reads with replacement, $B=200$ bootstrap replicates were
generated from a single real yeast RNA-seq library and re-quantified with `kallisto`. The
bootstrap standard errors of the transcript proportions match the analytic multinomial
standard errors to within a few percent, and the standardized bootstrap deviations follow
the standard normal distribution. The bootstrap standard errors are marginally larger than
the analytic errors, consistent with a small additional contribution from the
probabilistic assignment of reads compatible with more than one transcript.

The bootstrap also answers a question that has no closed-form solution: whether a
low-abundance transcript call is robust to the particular reads that were sequenced.
Detection is a discrete presence/absence event, so the per-transcript standard error of
panel B does not describe it. Panel C instead assigns each transcript a bootstrap
detection probability $\pi_t$ and reveals a band of fragile calls (here 287 transcripts at
a threshold of $\theta = 5$ estimated counts) whose presence depends on the particular
reads sequenced, some called at baseline yet vanishing in some replicates and some absent
at baseline yet appearing in others. This is the canonical use case for read-level
bootstrapping: propagating the sampling uncertainty of sequencing into the uncertainty of
any downstream estimate, including statistics such as a discrete detection call for which
no analytic standard error exists. It is only possible because `fastQpick` supports
sampling with replacement, which most FASTQ subsampling tools do not.